In [1]:
from minerva.models.nets.image.vit import mae_vit_base_patch16
import torch
from torch.utils.data import DataLoader, TensorDataset
import lightning as L

/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
num_samples = 8             # Number of samples in the dataset
batch_size = 2              # Number of samples per batch
data_shape = (1, 224, 224)  # Channels x Height x Width


train_X = torch.rand(num_samples, *data_shape)
train_y = torch.rand(num_samples, 1)        # Not used, but required for the dataset
print(f"Train X shape: {train_X.shape}. Train y shape: {train_y.shape}")

train_dataset = TensorDataset(train_X, train_y)
print(f"Number of samples of dataset: {len(train_dataset)}")

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
print(f"Number of batches: {len(train_dataloader)}")

first_batch = next(iter(train_dataloader))
print(f"First batch shape: x={first_batch[0].shape}, y={first_batch[1].shape}")

Train X shape: torch.Size([8, 1, 224, 224]). Train y shape: torch.Size([8, 1])
Number of samples of dataset: 8
Number of batches: 4
First batch shape: x=torch.Size([2, 1, 224, 224]), y=torch.Size([2, 1])


In [3]:
model = mae_vit_base_patch16()
model

MaskedAutoencoderViT(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(1, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop2

In [4]:
trainer = L.Trainer(
    max_epochs=1,
    accelerator="gpu",
    devices=1,
    logger=False,
    enable_checkpointing=False,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [5]:
trainer.fit(model, train_dataloader)

/usr/local/lib/python3.10/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



  | Name           | Type       | Params | Mode 
------------------------------------------------------
0 | patch_embed    | PatchEmbed | 197 K  | train
1 | blocks         | ModuleList | 85.1 M | train
2 | norm           | LayerNorm  | 1.5 K  | train
3 | decoder_embed  | Linear     | 393 K  | train
4 | decoder_blocks | ModuleList | 25.2 M | train
5 | decoder_norm   | LayerNorm  | 1.0 K  | train
6 | decoder_pred   | Linear     | 131 K  | train
  | other params   | n/a        | 253 K  | n/a  
------------------------------------------------------
110 M     Trainable params
252 K     Non-trainable params
111 M     Total params
445.008   Total estimated model params size (MB)
429       Modules in train mode
0         Modules in eval mode
/usr/local/lib/python3.10/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=2

Epoch 0: 100%|██████████| 4/4 [00:01<00:00,  3.70it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 4/4 [00:01<00:00,  3.68it/s]
